# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanBuilds/Rayanflyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring.** The transparent rule my model has to beat. No fitted
weights, reason codes on every row, precision@K reported next to the base rate.

Continues from `w03_data_contract.ipynb`. The rule uses **only** fields the contract bucketed as
features.

In [1]:
# Setup: work from the repo root so the starter CSV loads on Colab AND from a fresh local clone.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/KhanBuilds/Rayanflyrank"
REPO_DIR = "Rayanflyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), f"starter CSV not found at {DATA_PATH} - are you at the repo root?"

import numpy as np
import pandas as pd

RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
# The label is DEFINED from trend_direction, which is computed from the 30-day impression pair.
# So trend_direction, trend_pct, impressions_last_30d and impressions_prev_30d are never features.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
y = df["is_declining_label"].to_numpy()
BASE_RATE = float(y.mean())


def precision_at_k(labels, scores, k: int) -> float:
    """Share of genuine positives among the k highest-scoring rows."""
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())


print("Working dir:", os.getcwd())
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns; {df['client_id'].nunique()} clients.")
print(f"Label base rate: {BASE_RATE:.4f}")


Working dir: /content/Rayanflyrank
Loaded 30,000 rows x 45 columns; 32 clients.
Label base rate: 0.5421


## 1. My rule and its reason codes

**The rule in plain words.** A page is worth an editor's time first if it has **established but
imperfect search coverage**, **enough demand to be worth the hours**, sits at a **middling position**
(good enough to be findable, not so good that it's already winning), and is **stale and mature enough
that a refresh is plausible**.

That is deliberately five readable conditions, each worth points, no fitted weights:

| Condition | Points | Plain reading |
|---|---|---|
| `20 ≤ days_with_impressions ≤ 87` | **3** | The page has real, sustained coverage but is not showing every single day. |
| `impressions_90d ≥ 40` | **2** | There is enough demand for a refresh to be worth anything. |
| `3 < avg_position ≤ 50` | **2** | Findable but not already top-3. Excludes `avg_position == 0` (no data). |
| `days_since_last_update ≥ 90` | **1** | Not touched in a quarter. |
| `90 ≤ content_age_days < 365` | **1** | Old enough to have settled, young enough to still matter. |

Score range 0–9. Ties are broken by `min(log1p(impressions_90d), 10)` — bigger pages first within a
score band. **Section 4 shows this tie-break is the weakest part of the design.**

**Reason codes.** Every scored row carries the codes that fired, so the queue is auditable by a human:
`established_coverage`, `has_demand`, `mid_position`, `stale_90d`, `mature_page`. A row with no codes
carries `no_signal`.

**Where the thresholds came from — disclosed, because it matters.** I did not invent these cut points in
a vacuum: I read them off the label-rate crosstabs in `w02_ml_task_framing.ipynb` (the
`days_with_impressions` band peaking at 46–69 days, `striking`/`page_1` positions declining most). That
is **mild in-sample tuning**, and it means this baseline is fitted to the same 30,000 rows it is scored
on. Two honest consequences:

1. Its precision@K here is **optimistic** — a genuinely fresh-eyes rule would do somewhat worse.
2. So ML-08 must compare model vs baseline on a **client-held-out split**, scoring *both* on the same
   held-out clients. Comparing a tuned-on-everything rule against a properly held-out model would
   flatter the rule, not the model.

**The rule deliberately does NOT use** `trend_direction`, `trend_pct`, `impressions_last_30d`,
`impressions_prev_30d` (label-derived — see the contract), nor `content_id`/`client_id` (context), nor
`provider_used`/`model_used` (product flags). Section 4 asserts this in code.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# The rule, as five readable conditions. No fitted weights.
BASELINE_INPUTS = [
    "days_with_impressions", "impressions_90d", "avg_position",
    "days_since_last_update", "content_age_days",
]

# avg_position == 0 means "no data", not rank 0 -> treat as missing so it cannot satisfy mid_position.
position = df["avg_position"].replace(0, np.nan)

# Five readable conditions -> (boolean flag, points). Nothing is fitted.
RULE = {
    "established_coverage": (df["days_with_impressions"].between(20, 87), 3),
    "has_demand":           (df["impressions_90d"] >= 40, 2),
    "mid_position":         (((position > 3) & (position <= 50)).fillna(False), 2),
    "stale_90d":            (df["days_since_last_update"] >= 90, 1),
    "mature_page":          (df["content_age_days"].between(90, 364), 1),
}

baseline = df[["content_id", "client_id", "is_declining_label"] + BASELINE_INPUTS + ["ctr", "content_type"]].copy()
baseline["baseline_score"] = sum(flag.astype(int) * pts for flag, pts in RULE.values())

# Reason codes: which conditions fired on this row, so a human can audit the pick.
flag_frame = pd.DataFrame({name: flag.to_numpy() for name, (flag, _) in RULE.items()})
baseline["reason_codes"] = [
    ",".join(flag_frame.columns[row]) or "no_signal" for row in flag_frame.to_numpy()
]

print("The rule (condition -> points):")
for name, (flag, pts) in RULE.items():
    print(f"  {name:22s} +{pts}   fires on {flag.sum():,} rows ({flag.mean():.1%})")
print()

print(f"score range: {baseline['baseline_score'].min()} - {baseline['baseline_score'].max()}")
print()
print("Label rate by score level (is the score ordered the way I claim?):")
print(baseline.groupby("baseline_score")["is_declining_label"].agg(["size", "mean"]).round(3).to_string())
print()
print(f"Base rate for comparison: {BASE_RATE:.3f}")

The rule (condition -> points):
  established_coverage   +3   fires on 13,078 rows (43.6%)
  has_demand             +2   fires on 23,959 rows (79.9%)
  mid_position           +2   fires on 26,340 rows (87.8%)
  stale_90d              +1   fires on 9,345 rows (31.1%)
  mature_page            +1   fires on 23,640 rows (78.8%)

score range: 0 - 9

Label rate by score level (is the score ordered the way I claim?):
                size   mean
baseline_score             
0                 75  0.293
1               1809  0.088
2                612  0.350
3               2785  0.417
4               3356  0.449
5               4331  0.577
6               5145  0.593
7               2606  0.467
8               6291  0.684
9               2990  0.714

Base rate for comparison: 0.542


## 2. Build the ranked queue (writes the CSV)

Rank every page, attach the reason codes, write the queue and the metrics.

Two files come out of this cell:

- `work/outputs/baseline_action_score.csv` — the full ranked queue (30,000 rows). Gitignored by design
  (`work/**/*.csv` in `.gitignore`), because datasets never enter git.
- `work/outputs/baseline_metrics.json` — **committed**, deliberately. It is the receipt every
  precision@K number in my capstone report traces back to. If a number in the report and a number in
  this file disagree, the report is wrong.

**Result, stated with its base rate:** precision@50 = **0.740** against a base rate of **0.542**. So of
50 slots, about 37 land on pages measured as declining, where random triage would land about 27. That is
the bar ML-08 has to beat — and note it already matches the reference pipeline's *random forest*
(0.740), while beating the reference *rule* baseline (0.240) by a wide margin, because that rule keys on
staleness which barely exists in this slice (Section 4).

**Precision is not monotone in K**, which is a real finding rather than noise: 0.750 at K=20, 0.740 at
K=50, then **0.800 at K=100** and 0.835 at K=200. A ranking that gets *better* deeper into the list means
the ordering inside the top band is wrong — diagnosed in Section 4.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
import json
from pathlib import Path

# Tie-break inside a score band: bigger pages first (capped so one huge page cannot dominate).
baseline["rank_key"] = baseline["baseline_score"] * 100 + np.minimum(np.log1p(baseline["impressions_90d"]), 10)
queue = baseline.sort_values("rank_key", ascending=False, kind="stable").reset_index(drop=True)
queue.insert(0, "queue_rank", np.arange(1, len(queue) + 1))

y_queue = queue["is_declining_label"].to_numpy()
K_VALUES = [20, 50, 100, 200, 500]
metrics = {f"precision_at_{k}": round(float(y_queue[:k].mean()), 4) for k in K_VALUES}
metrics.update({
    "base_rate": round(BASE_RATE, 4),
    "rows_scored": int(len(queue)),
    "positives": int(y_queue.sum()),
    "random_state": RANDOM_STATE,
    "evaluation": "full-data, in-sample (thresholds read off w02 crosstabs) - not a held-out estimate",
    "rule": "3*established_coverage + 2*has_demand + 2*mid_position + 1*stale_90d + 1*mature_page",
})

print(f"base rate                {metrics['base_rate']:.4f}   <- the number every precision sits next to")
for k in K_VALUES:
    p = metrics[f"precision_at_{k}"]
    print(f"precision@{k:<4d}           {p:.4f}   ({p / metrics['base_rate']:.2f}x base rate, "
          f"{int(round(p * k))}/{k} slots on genuinely declining pages)")

out_dir = Path("work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)
queue_cols = ["queue_rank", "content_id", "client_id", "baseline_score", "reason_codes",
              "impressions_90d", "days_with_impressions", "avg_position", "ctr",
              "days_since_last_update", "content_age_days", "content_type", "is_declining_label"]
queue[queue_cols].to_csv(out_dir / "baseline_action_score.csv", index=False)
(out_dir / "baseline_metrics.json").write_text(json.dumps(metrics, indent=2, sort_keys=True))
print(f"\nwrote work/outputs/baseline_action_score.csv  ({len(queue):,} rows, gitignored)")
print("wrote work/outputs/baseline_metrics.json     (committed - the receipt for these numbers)")

base rate                0.5421   <- the number every precision sits next to
precision@20             0.7500   (1.38x base rate, 15/20 slots on genuinely declining pages)
precision@50             0.7400   (1.37x base rate, 37/50 slots on genuinely declining pages)
precision@100            0.8000   (1.48x base rate, 80/100 slots on genuinely declining pages)
precision@200            0.8350   (1.54x base rate, 167/200 slots on genuinely declining pages)
precision@500            0.8080   (1.49x base rate, 404/500 slots on genuinely declining pages)

wrote work/outputs/baseline_action_score.csv  (30,000 rows, gitignored)
wrote work/outputs/baseline_metrics.json     (committed - the receipt for these numbers)


## 3. Top-20 review

Hand-reviewed. All 20 sit at the maximum score of 9, so what actually orders them is the impressions
tie-break — and that is where the errors are.

**Result: 15 of 20 are genuinely declining (0.750).** The 5 misses are *not* random: **3 are `up`**
(+54.4%, +68.5%, +35.2% impression change) and **2 are `stable`** (+0.1%, +3.0%).

**The damning detail: the three highest-ranked pages in my queue are all growing.** Ranks 1, 2 and 3 are
the three biggest pages in the top band, and big pages in this data are *more likely to be growing*.
My tie-break puts them first. So the rule identifies the right *band* and then orders it backwards at
the very top.

| Rank | Action | Reason codes | Confidence | What would make it wrong |
|---|---|---|---|---|
| 1–3 | **Do not send to an editor** — monitor only | all five codes fired | **Low.** Highest impressions in the band, and impressions are anti-correlated with decline here | Already wrong: all three measured `up` (+35% to +69%). The tie-break, not the rule, put them here |
| 4–13 | Refresh review | all five codes | **Medium-high.** 10/10 declining, drops of −21% to −96% | Off-season intent (an autumn topic measured in summer) would look identical; the data has no seasonality field to rule it out |
| 14, 17 | Monitor, don't refresh | all five codes | **Low.** Both measured `stable` (+0.1%, +3.0%) | Nothing to fix — these are flat pages the rule cannot distinguish from decaying ones |
| 15, 16, 18–20 | Refresh review | all five codes | **Medium.** 5/5 declining, drops of −40% to −95% | Rank 19 is a `feedly article`; that content type is 46.6% `new` and structurally unlikely to be labelled declining, so a hit here is partly luck |

**Two operational observations a metric table would hide:**

1. **The queue is client-concentrated.** The top 20 draws from only **6 of 32 clients**, and one client
   supplies **35%** of it (top 50: 8 clients, one supplying 44%). An editor serving all 32 clients would
   get a sprint queue that ignores three quarters of the portfolio. A production version needs a
   per-client cap, which is a *product* decision my metric never sees.
2. **The staleness signal is really a client cadence artifact.** **17 of the top 20** have
   `days_since_last_update == 104` — the same value. That is one client's bulk-update rhythm, not 17
   independent editorial judgements. The `stale_90d` point is therefore doing much less independent work
   than the rule's design implies.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
top20 = queue.head(20)
view = ["queue_rank", "baseline_score", "impressions_90d", "days_with_impressions",
        "avg_position", "ctr", "days_since_last_update", "content_age_days",
        "content_type", "is_declining_label"]
print("Top 20 (client_id and content_id withheld from the printout - public-safe):")
print(top20[view].to_string(index=False))
print()
print(f"top-20 precision: {top20['is_declining_label'].mean():.3f}  (base rate {BASE_RATE:.3f})")
print(f"all rows share the same reason codes: {top20['reason_codes'].nunique() == 1} "
      f"-> ordering is driven purely by the impressions tie-break")
print()

misses = top20[top20["is_declining_label"] == 0]
miss_trend = df.loc[misses.index if misses.index.isin(df.index).all() else [], :]
print(f"misses in the top 20: {len(misses)} of 20")
merged = misses.merge(df[["content_id", "trend_direction", "trend_pct"]], on="content_id", how="left")
print("what the misses actually did (label-derived columns, used ONLY for this post-hoc review):")
print(merged[["queue_rank", "impressions_90d", "trend_direction", "trend_pct"]].to_string(index=False))
print()

print("Queue concentration by client (counts only, no client identifiers printed):")
for k in (20, 50, 100):
    head = queue.head(k)
    counts = head["client_id"].value_counts()
    print(f"  top {k:<4d}: {counts.size} distinct clients | largest single client = {counts.iloc[0] / k:.0%} of the queue")
print()
print("Staleness values in the top 20 (one client's bulk-update cadence, not 20 judgements):")
print(top20["days_since_last_update"].value_counts().to_string())

Top 20 (client_id and content_id withheld from the printout - public-safe):
 queue_rank  baseline_score  impressions_90d  days_with_impressions  avg_position  ctr  days_since_last_update  content_age_days    content_type  is_declining_label
          1               9            12863                     79          19.7 0.79                     104               229 keyword article                   0
          2               9            10356                     87           8.0 0.14                     104               313 keyword article                   0
          3               9             9612                     86           6.3 0.12                     104               153 keyword article                   0
          4               9             9345                     86          14.5 0.05                     104               211 keyword article                   1
          5               9             8915                     83          32.2 0.00             

## 4. Weak picks + leakage check

### The weak picks, named

**1. The tie-break is backwards at the top.** Precision climbs with K — 0.750 @20, 0.740 @50, **0.800
@100**, 0.835 @200 — which can only happen if the highest-ranked rows are worse than the ones just
below. Cause: within the max-score band I order by impressions, and impressions are mildly
*anti*-correlated with this label (the biggest pages are the most stable). Measured alternative below:
reversing the tie-break makes the very top no better (0.750 @20) and everything deeper worse (0.720 @50,
0.680 @100), so I keep the current tie-break and report the flaw rather than pretending the ordering is
principled. **The honest reading: my score identifies a good band (2,990 pages at score 9, 71.4%
declining) but does not meaningfully rank inside it.** That is precisely the gap a model can fill.

**2. The score is not monotone.** Label rate by score level climbs from 0.088 (score 1) to 0.714 (score
9), but with two inversions: score 1 (0.088, n=1,809) sits *below* score 0 (0.293, n=75), and score 7
(0.467, n=2,606) sits *below* score 6 (0.593, n=5,145). The additive weights are not a calibrated
ordering — they are five opinions added together, and "8 points" does not reliably mean "worse off than
7 points".

**3. The rule cannot see seasonality or intent shifts**, which is the most likely real-world cause of a
false positive. There is no seasonality field in the data, so I cannot even measure how often this
happens — I can only name it as an unquantified error mode.

**4. The reference rule baseline's 0.240 is a tie-break artifact, and mine could have been one too.**
Demonstrated below: the textbook "stale ≥180d AND ≥500 impressions" rule fires on **17 rows**, then
scores zero for everyone else, so ranks 18–50 are filled in arbitrary file order. Its precision@50
"0.740" is 17 real picks plus 33 coin flips. My rule's max-score band holds 2,990 rows, so the top 50 is
genuinely selected — this is the difference between a headline number and a headline artifact.

### Leakage check

Asserted in code below, not asserted in prose:

- The score reads exactly **5 input columns**, and none of them is `trend_direction`, `trend_pct`,
  `impressions_last_30d`, `impressions_prev_30d` or `is_declining_label`.
- No `client_id` / `content_id` enters the score (context only — they appear in the output for grouping
  and audit, never in the arithmetic).
- No future window: every input is a trailing-90-day or age/offset field. There is no "after" period in
  this file to leak from.
- **Reconstruction test:** if the score were secretly label-derived, a rule this simple would separate
  the classes far better than it does. Measured ROC-AUC of the raw score is reported below — a modest
  number is itself evidence of no leak. A near-1.0 AUC from five hand-written conditions would have been
  the alarm.

`trend_direction` and `trend_pct` **do** appear in Section 3 — used post-hoc to describe what the misses
actually did. That is evaluation, not scoring, and the code above touches them only after the queue is
frozen.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one - typing sentences here breaks Run All.
# --- LEAKAGE ASSERTIONS ---------------------------------------------------
FORBIDDEN = {"trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
             "is_declining_label", "client_id", "content_id"}
assert not FORBIDDEN & set(BASELINE_INPUTS), "a forbidden column reached the score"
print(f"score inputs ({len(BASELINE_INPUTS)}): {BASELINE_INPUTS}")
print(f"forbidden columns among them: {sorted(FORBIDDEN & set(BASELINE_INPUTS)) or 'none'}")

# Rank-correlation sanity check: a leaky score would separate the classes almost perfectly.
def roc_auc(labels, scores) -> float:
    """AUC via the rank-sum identity - no sklearn needed."""
    labels = np.asarray(labels)
    ranks = pd.Series(scores).rank().to_numpy()
    n_pos, n_neg = labels.sum(), (1 - labels).sum()
    return float((ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

auc = roc_auc(y_queue, queue["baseline_score"])
print(f"\nbaseline score ROC-AUC: {auc:.4f}  (modest = consistent with no leakage; ~1.0 would be an alarm)")
assert auc < 0.90, "suspiciously high AUC for five hand-written conditions - check for leakage"
print()

# --- WEAK PICK 1: the tie-break is backwards at the top -------------------
print("Tie-break comparison (same score, different ordering inside each band):")
alt = baseline.copy()
alt["rank_key_rev"] = alt["baseline_score"] * 100 - np.minimum(np.log1p(alt["impressions_90d"]), 10)
alt_queue = alt.sort_values("rank_key_rev", ascending=False, kind="stable")
y_alt = alt_queue["is_declining_label"].to_numpy()
print(f"  {'K':>5} {'impressions desc':>18} {'impressions asc':>18}")
for k in (20, 50, 100, 200):
    print(f"  {k:>5} {y_queue[:k].mean():>18.3f} {y_alt[:k].mean():>18.3f}")
print("  -> precision RISES with K under both orderings: the band is good, the ordering inside it is not")
print()

# --- WEAK PICK 2: the score is not monotone -------------------------------
by_score = baseline.groupby("baseline_score")["is_declining_label"].agg(["size", "mean"]).round(3)
by_score["monotone_break"] = by_score["mean"].diff() < 0
print("Label rate by score level (monotone_break = this level is worse than the one below it):")
print(by_score.to_string())
print()

# --- WEAK PICK 4: the naive stale rule is a tie-break artifact ------------
naive = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).astype(int)
naive_score = naive * df["impressions_90d"]
fires = int(naive.sum())
print(f"Naive 'stale >=180d AND >=500 impressions' rule fires on {fires} rows of {len(df):,}.")
print(f"  its precision@50 = {precision_at_k(y, naive_score, 50):.3f} "
      f"-> but only {fires} of those 50 were actually selected by the rule; "
      f"{50 - fires} are arbitrary file order")
print(f"  my rule's top band holds {(baseline['baseline_score'] == 9).sum():,} rows "
      f"({baseline.loc[baseline['baseline_score'] == 9, 'is_declining_label'].mean():.3f} declining), "
      f"so its top 50 is genuinely selected")

score inputs (5): ['days_with_impressions', 'impressions_90d', 'avg_position', 'days_since_last_update', 'content_age_days']
forbidden columns among them: none

baseline score ROC-AUC: 0.6485  (modest = consistent with no leakage; ~1.0 would be an alarm)

Tie-break comparison (same score, different ordering inside each band):
      K   impressions desc    impressions asc
     20              0.750              0.750
     50              0.740              0.720
    100              0.800              0.680
    200              0.835              0.675
  -> precision RISES with K under both orderings: the band is good, the ordering inside it is not

Label rate by score level (monotone_break = this level is worse than the one below it):
                size   mean  monotone_break
baseline_score                             
0                 75  0.293           False
1               1809  0.088            True
2                612  0.350           False
3               2785  0.417        

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.